# Notebook 05 — Model E: SBERT Text Fusion
## FYP: Adaptive Explainable Ensemble for Pre-Launch Steam Game Reception Prediction
### Heshan Ratnaweera | IIT Sri Lanka | W2082289 | 2026

> ⚠️ **Run this notebook on Google Colab with GPU enabled.**
> Runtime → Change runtime type → T4 GPU
> SBERT encoding 20,383 descriptions takes ~2 min on GPU vs ~35 min on CPU.

**Purpose:** Extend the T4 structured feature set with 50-dimensional SBERT text
embeddings derived from developer-written game descriptions, train Model E on the
combined 103-feature vector, and compare against Model D (structured only).

**Inputs:** `games_features_t4.csv` (from your Google Drive)
**Outputs (download back to local machine):**
- `sbert_embeddings_raw.npy` → `data/processed/`
- `sbert_embeddings_pca{N}.npy` → `data/processed/`  *(N = selected component count)*
- `pca_transformer.pkl` → `models/`
- `model_e.pkl` → `models/`

---
## Contents
1. Colab Setup & GPU Check
2. Mount Google Drive & Load Data
3. Train-Test Split (identical to notebooks 03 & 04)
4. Prepare Descriptions — Strip HTML
5. SBERT Encoding
6. PCA Dimensionality Reduction (no leakage)
7. Build T5 Feature Matrix
8. Train Model E
9. Three-Way Comparison: Model D vs TF-IDF vs Model E (SBERT)
10. Ablation — Description Coverage Impact
11. Save All Outputs
12. Summary & Download Instructions


## 1. Colab Setup & GPU Check

In [ ]:
# ── Install required packages ─────────────────────────────────────────────────
# sentence-transformers is not pre-installed on Colab
# catboost and joblib are pre-installed on Colab but we confirm
import subprocess
subprocess.run(['pip', 'install', 'sentence-transformers', '-q'])
subprocess.run(['pip', 'install', 'catboost', '-q'])

print('Packages installed.')

# ── GPU check ─────────────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'GPU available: {gpu_name}')
    print(f'CUDA version : {torch.version.cuda}')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU detected. Encoding will be slow (~35 min).')
    print('Go to Runtime → Change runtime type → T4 GPU and reconnect.')
    DEVICE = 'cpu'

print(f'Using device: {DEVICE}')


## 2. Mount Google Drive & Load Data

Upload `games_features_t4.csv` to your Google Drive before running this cell.
Recommended path: `My Drive/fyp_pprs/games_features_t4.csv`

If you used a different path, update `CSV_PATH` below.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import warnings
import time
warnings.filterwarnings('ignore')

# ── Update this path if you saved the CSV elsewhere in Google Drive ───────────
CSV_PATH = '/content/drive/MyDrive/fyp_pprs/games_features_t4.csv'

df = pd.read_csv(CSV_PATH)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

# ── Constants (must match notebooks 03 and 04) ────────────────────────────────
TARGET_COL    = 'reception_label'
DESC_COL      = 'short_description'
RANDOM_STATE  = 42
TEST_SIZE     = 0.2
CV_FOLDS      = 5
PCA_COMPONENTS = 50
SBERT_MODEL   = 'all-MiniLM-L6-v2'

# ── UPDATE THIS after running NB03 with the expanded iterations search ────────
# Set to the best iterations value found by NB03 (200, 500, or 1000)
BEST_ITERATIONS = 1000  # confirmed best iterations from NB03 tuning (depth/lr/l2 unchanged)

# ── T4 feature columns (53 structured features — same as notebook 04) ─────────
EXCLUDE_COLS  = [TARGET_COL, DESC_COL, 'dlc_count']
T4_FEATURES   = [c for c in df.columns if c not in EXCLUDE_COLS]

print(f'T4 structured features: {len(T4_FEATURES)}')
print(f'Class balance: {df[TARGET_COL].mean()*100:.1f}% positive')
print(f'Description column: {DESC_COL}')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Train-Test Split (Identical to Notebooks 03 & 04)

**Critical:** Same `RANDOM_STATE=42` and parameters as all previous notebooks.
This ensures the test set contains exactly the same games, making cross-notebook
comparisons valid.


In [ ]:
from sklearn.model_selection import train_test_split

# Split indices — used to slice all arrays (features, descriptions, embeddings)
all_indices = np.arange(len(df))
train_idx, test_idx = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,   # MUST match notebooks 03 and 04
    stratify=df[TARGET_COL].values
)

y       = df[TARGET_COL].values
y_train = y[train_idx]
y_test  = y[test_idx]

X_t4_train = df.iloc[train_idx][T4_FEATURES].values
X_t4_test  = df.iloc[test_idx][T4_FEATURES].values

print(f'Train set: {len(train_idx):,} games  ({y_train.mean()*100:.1f}% positive)')
print(f'Test set : {len(test_idx):,} games   ({y_test.mean()*100:.1f}% positive)')
print(f'T4 feature matrix: train={X_t4_train.shape}  test={X_t4_test.shape}')


## 4. Prepare Descriptions — Strip HTML

Steam descriptions often contain HTML tags (`<br>`, `<b>`, `<i>`, etc.).
We strip these to get clean plain text before encoding with SBERT.

37 games have no description after stripping — these will be handled in section 10.


In [ ]:
try:
    from bs4 import BeautifulSoup
except ImportError:
    subprocess.run(['pip', 'install', 'beautifulsoup4', '-q'])
    from bs4 import BeautifulSoup

def strip_html(text):
    """Remove HTML tags and return clean plain text. Returns '' for null/empty."""
    if pd.isna(text) or str(text).strip() == '':
        return ''
    return BeautifulSoup(str(text), 'html.parser').get_text().strip()

print('Stripping HTML from descriptions...')
descriptions = df[DESC_COL].apply(strip_html).values

# Coverage check
n_empty   = (descriptions == '').sum()
n_total   = len(descriptions)
coverage  = (n_total - n_empty) / n_total * 100

print(f'Total games            : {n_total:,}')
print(f'With description       : {n_total - n_empty:,}  ({coverage:.1f}%)')
print(f'Without description    : {n_empty:,}  ({100-coverage:.1f}%) → fall back to Model D')
print()

# Sample descriptions to verify
print('Sample descriptions (first 3 non-empty):')
samples = [(i, d) for i, d in enumerate(descriptions) if d][:3]
for i, d in samples:
    print(f'  Game {i}: {d[:100]}...')


## 5. SBERT Encoding

`all-MiniLM-L6-v2` encodes each description into a 384-dimensional vector.
This model was chosen because it:
- Is optimised for semantic similarity tasks
- Produces strong sentence-level representations
- Is fast and lightweight (22M parameters)
- Achieves excellent performance on the SBERT benchmark

Games with no description receive a zero vector — these are handled explicitly
in section 10.


In [ ]:
from sentence_transformers import SentenceTransformer

print(f'Loading SBERT model: {SBERT_MODEL}')
sbert_model = SentenceTransformer(SBERT_MODEL, device=DEVICE)
print(f'Model loaded. Embedding dimension: {sbert_model.get_sentence_embedding_dimension()}')
print()

# ── Encode all descriptions ────────────────────────────────────────────────────
# Games with no description get a zero vector
# The zero vector is semantically neutral — it adds no signal for missing games
print(f'Encoding {n_total:,} descriptions on {DEVICE.upper()}...')
start = time.time()

# Replace empty descriptions with a neutral placeholder for encoding
# We will zero these out afterward
desc_for_encoding = ['No description available' if d == '' else d
                     for d in descriptions]

embeddings_raw = sbert_model.encode(
    desc_for_encoding,
    batch_size=256,          # 256 per GPU batch — adjust down to 128 if memory error
    show_progress_bar=True,
    convert_to_numpy=True,
    device=DEVICE,
    normalize_embeddings=False
)

elapsed = time.time() - start
print(f'\nEncoding complete in {elapsed:.1f}s')
print(f'Embedding matrix shape: {embeddings_raw.shape}')  # should be (20383, 384)

# Zero out embeddings for games with no description
empty_mask = (descriptions == '')
embeddings_raw[empty_mask] = 0.0
print(f'Zeroed out {empty_mask.sum()} embeddings for games with no description')

# Save raw embeddings immediately — this took time to compute
np.save('/content/sbert_embeddings_raw.npy', embeddings_raw)
print('Saved: /content/sbert_embeddings_raw.npy')


## 6a. PCA Component Search — Empirical Selection by Model Performance

Instead of choosing PCA components based on variance alone, we test each
candidate value empirically — training a mini Model E for each and measuring
CV macro F1 on the training set.

**Why empirical selection?**
Variance retained tells us how much SBERT information survives PCA compression.
But more variance does not always mean better model performance. More PCA
dimensions can introduce noise that hurts CatBoost. We search for the component
count that actually maximises macro F1, not just variance.

**Two-stage process:**
1. Variance scan — confirm each candidate retains meaningful information
2. Model performance scan — train Model E for each and pick the best macro F1

The winning `PCA_COMPONENTS` value is used throughout the rest of the notebook.


In [ ]:
from sklearn.decomposition import PCA as _PCA
from catboost import CatBoostClassifier as _CB
from sklearn.model_selection import cross_val_score, StratifiedKFold as _SKF
from sklearn.metrics import make_scorer, f1_score
import time as _time

# ── Stage 1: Variance scan ────────────────────────────────────────────────────
CANDIDATE_COMPONENTS = [50, 100, 150, 200, 250, 300]
VARIANCE_TARGET = 90.0

print('STAGE 1 — Variance scan')
print(f'  Target: ≥{VARIANCE_TARGET}% variance retained')
print()

variance_results = {}
for n in CANDIDATE_COMPONENTS:
    p = _PCA(n_components=n, random_state=42)
    p.fit(embeddings_raw[train_idx])
    var = p.explained_variance_ratio_.sum() * 100
    variance_results[n] = var
    meets = '✓' if var >= VARIANCE_TARGET else '✗'
    print(f'  PCA({n:>3}): variance = {var:.1f}%  {meets}')

print()

# ── Stage 2: Empirical model performance scan ─────────────────────────────────
# For each candidate, build the T5 matrix and measure CV macro F1
# We use 3-fold CV here (not 5) to keep the search fast
# The final model uses 5-fold CV in section 8

print('STAGE 2 — Model performance scan (3-fold CV on training set)')
print('  This trains a CatBoost for each PCA component count...')
print()

_cv3 = _SKF(n_splits=3, shuffle=True, random_state=42)
_f1_macro = make_scorer(f1_score, average='macro')

def _build_quick_catboost():
    return _CB(
        iterations=100,          # fewer iterations for speed in the search
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3,
        scale_pos_weight=0.5,
        random_seed=42,
        verbose=0,
        task_type='GPU' if DEVICE == 'cuda' else 'CPU'
    )

performance_results = {}
print(f'  {"PCA(n)":>8}  {"Variance":>10}  {"CV Macro F1":>12}  {"Time":>8}')
print('  ' + '-' * 46)

for n in CANDIDATE_COMPONENTS:
    _start = _time.time()

    # Fit PCA on train only
    _pca_tmp = _PCA(n_components=n, random_state=42)
    _emb_train_n = _pca_tmp.fit_transform(embeddings_raw[train_idx])

    # Build T5 matrix for this component count
    _X_t5_n = __import__('numpy').hstack([X_t4_train, _emb_train_n])

    # 3-fold CV macro F1
    _scores = cross_val_score(
        _build_quick_catboost(), _X_t5_n, y_train,
        cv=_cv3, scoring=_f1_macro, n_jobs=1
    )
    _macro = _scores.mean()
    _elapsed = _time.time() - _start

    performance_results[n] = _macro
    var = variance_results[n]
    print(f'  PCA({n:>3}):  var={var:>6.1f}%  macro_f1={_macro:.4f}  time={_elapsed:.1f}s')

# ── Select best component count ───────────────────────────────────────────────
print()
PCA_COMPONENTS = max(performance_results, key=performance_results.get)
best_macro     = performance_results[PCA_COMPONENTS]
best_variance  = variance_results[PCA_COMPONENTS]

print('SELECTION SUMMARY:')
print(f'  {"n":>6}  {"Variance":>10}  {"CV Macro F1":>12}')
print('  ' + '-' * 34)
for n in CANDIDATE_COMPONENTS:
    marker = ' ← SELECTED' if n == PCA_COMPONENTS else ''
    print(f'  {n:>6}  {variance_results[n]:>9.1f}%  {performance_results[n]:>12.4f}{marker}')

print()
print(f'Selected PCA_COMPONENTS = {PCA_COMPONENTS}')
print(f'  Variance retained : {best_variance:.1f}%')
print(f'  CV Macro F1 (3-fold): {best_macro:.4f}')
print(f'  T5 feature vector : 53 structured + {PCA_COMPONENTS} text = {53 + PCA_COMPONENTS} total')


## 6. PCA Dimensionality Reduction — No Leakage

We reduce 384 SBERT dimensions to a smaller number using PCA, with the exact
component count determined automatically in section 6a above.

**Critical — no data leakage:** PCA must be fit on the **training set only**.
Fitting on the full dataset would allow information from the test set to influence
the transformation, giving an optimistic performance estimate.

The fitted PCA transformer is saved — it will be used at inference time in the
Streamlit app to transform new game descriptions before prediction.

The component count is selected to retain ≥90% of the embedding variance.
An initial run with 50 components retained only 51.1% variance, causing Model E
to underperform Model D. The automated search in section 6a identified 200
components (91.4% variance) as the optimal value for this dataset.


In [ ]:
import joblib
from sklearn.decomposition import PCA

# ── PCA_COMPONENTS was selected empirically in section 6a ─────────────────────
print(f'Using PCA_COMPONENTS = {PCA_COMPONENTS}  ({53 + PCA_COMPONENTS} total T5 features)')
print()

# ── Split embeddings into train and test ──────────────────────────────────────
embeddings_train = embeddings_raw[train_idx]
embeddings_test  = embeddings_raw[test_idx]

# ── Fit PCA on TRAIN SPLIT ONLY — prevents data leakage ──────────────────────
print(f'Fitting PCA({PCA_COMPONENTS}) on training split only ({len(train_idx):,} games)...')
pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
pca.fit(embeddings_train)

variance_retained = pca.explained_variance_ratio_.sum() * 100
print(f'Variance retained by {PCA_COMPONENTS} components: {variance_retained:.1f}%')
print()

# Plot explained variance curve
import matplotlib.pyplot as plt
cumvar = __import__('numpy').cumsum(pca.explained_variance_ratio_) * 100
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, PCA_COMPONENTS + 1), cumvar, '-', color='#3b82f6', linewidth=1.5)
ax.axhline(90, color='orange', linestyle='--', linewidth=1, label='90% variance')
ax.axhline(95, color='red',    linestyle='--', linewidth=1, label='95% variance')
ax.axvline(PCA_COMPONENTS, color='gray', linestyle='--', linewidth=1,
           label=f'{PCA_COMPONENTS} components ({variance_retained:.1f}%)')
ax.set_xlabel('Number of PCA Components')
ax.set_ylabel('Cumulative Explained Variance (%)')
ax.set_title(f'PCA Explained Variance — SBERT Embeddings (fit on training set only)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/content/05_pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/05_pca_variance.png')

# ── Transform both splits ─────────────────────────────────────────────────────
embeddings_pca_train = pca.transform(embeddings_train)
embeddings_pca_test  = pca.transform(embeddings_test)

print(f'\nPCA output shapes:')
print(f'  Train: {embeddings_pca_train.shape}')
print(f'  Test : {embeddings_pca_test.shape}')

# ── Save PCA-reduced matrix for all games ─────────────────────────────────────
# Filename includes component count to avoid confusion with earlier runs
embeddings_pca_full = pca.transform(embeddings_raw)
PCA_NPY_FILENAME = f'sbert_embeddings_pca{PCA_COMPONENTS}.npy'
__import__('numpy').save(f'/content/{PCA_NPY_FILENAME}', embeddings_pca_full)
print(f'Saved: /content/{PCA_NPY_FILENAME}  shape={embeddings_pca_full.shape}')

# Save fitted PCA transformer for inference
joblib.dump(pca, '/content/pca_transformer.pkl')
print('Saved: /content/pca_transformer.pkl')


## 7. Build T5 Feature Matrix

The T5 matrix concatenates:
- **53 structured features** from T4 (price, genre one-hots, tags, categories, etc.)
- **50 SBERT PCA dimensions** from the text embeddings

Result: 103-dimensional feature vector per game.

Column naming: `text_dim_0` through `text_dim_49` for the PCA dimensions.


In [ ]:
# ── Concatenate structured + text features ────────────────────────────────────
# T5 = T4 structured features + PCA-reduced SBERT embeddings
# PCA_COMPONENTS was selected automatically in section 6a
X_t5_train = np.hstack([X_t4_train, embeddings_pca_train])
X_t5_test  = np.hstack([X_t4_test,  embeddings_pca_test])

print(f'T5 feature matrix (structured + SBERT PCA):')
print(f'  Train: {X_t5_train.shape}  ({X_t4_train.shape[1]} structured + {PCA_COMPONENTS} text dims)')
print(f'  Test : {X_t5_test.shape}')

# ── Verify no NaN or Inf values ───────────────────────────────────────────────
for name, arr in [('T5 train', X_t5_train), ('T5 test', X_t5_test)]:
    n_nan = np.isnan(arr).sum()
    n_inf = np.isinf(arr).sum()
    print(f'{name}: NaN={n_nan}, Inf={n_inf}  {"✓ clean" if n_nan == 0 and n_inf == 0 else "✗ PROBLEM"}')

# ── Feature column names for reference ────────────────────────────────────────
text_dim_names   = [f'text_dim_{i}' for i in range(PCA_COMPONENTS)]
T5_FEATURE_NAMES = T4_FEATURES + text_dim_names
print(f'\nTotal T5 feature names : {len(T5_FEATURE_NAMES)}')
print(f'  Structured ({len(T4_FEATURES)}): {T4_FEATURES[:3]} ...')
print(f'  Text dims  ({PCA_COMPONENTS}): {text_dim_names[:3]} ...')


## 8. Train Model E

Model E uses the same CatBoost hyperparameters as Models A–D for consistency.
This ensures performance differences between Model D and Model E reflect the
contribution of SBERT features, not algorithmic differences.

Best params from notebook 03:
`depth=8, learning_rate=0.05, iterations=200, l2_leaf_reg=3, scale_pos_weight=0.5`


In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import f1_score, roc_auc_score, make_scorer, classification_report, accuracy_score

f1_minority_scorer = make_scorer(f1_score, pos_label=0)
f1_majority_scorer = make_scorer(f1_score, pos_label=1)

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def build_catboost():
    """Same config as notebooks 03 and 04."""
    return CatBoostClassifier(
        iterations=BEST_ITERATIONS,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3,
        scale_pos_weight=0.5,
        random_seed=RANDOM_STATE,
        verbose=0,
        task_type='GPU' if DEVICE == 'cuda' else 'CPU'
        # task_type='GPU' uses the Colab GPU for CatBoost training too
    )

# ── 5-fold CV on training set ─────────────────────────────────────────────────
print('Training Model E (T5 — 103 features) with 5-fold CV...')
start = time.time()

clf_e = build_catboost()
scores_e = cross_validate(
    clf_e, X_t5_train, y_train,
    cv=cv,
    scoring={
        'f1_macro':    'f1_macro',
        'f1_minority': f1_minority_scorer,
        'f1_majority': f1_majority_scorer,
        'roc_auc':     'roc_auc',
        'accuracy':    'accuracy',   # NEW
    },
    n_jobs=1,
    return_train_score=False
)

elapsed = time.time() - start
print(f'CV complete in {elapsed:.1f}s')
print()
print(f'Model E CV Results:')
print(f'  Macro F1    : {scores_e["test_f1_macro"].mean():.4f} ± {scores_e["test_f1_macro"].std():.4f}')
print(f'  Minority F1 : {scores_e["test_f1_minority"].mean():.4f}')
print(f'  Majority F1 : {scores_e["test_f1_majority"].mean():.4f}')
print(f'  AUC-ROC     : {scores_e["test_roc_auc"].mean():.4f}')
print(f'  Accuracy    : {scores_e["test_accuracy"].mean():.4f} ± {scores_e["test_accuracy"].std():.4f}')

# ── Final fit on full training set ────────────────────────────────────────────
print()
print('Fitting Model E on full training set...')
clf_e_final = build_catboost()
clf_e_final.fit(X_t5_train, y_train)
print('Done.')

# ── Test set evaluation ───────────────────────────────────────────────────────
y_pred_e = clf_e_final.predict(X_t5_test)
y_prob_e = clf_e_final.predict_proba(X_t5_test)[:, 1]

test_macro_e = f1_score(y_test, y_pred_e, average='macro')
test_minor_e = f1_score(y_test, y_pred_e, pos_label=0)
test_major_e = f1_score(y_test, y_pred_e, pos_label=1)
test_auc_e      = roc_auc_score(y_test, y_prob_e)
test_accuracy_e = accuracy_score(y_test, y_pred_e)

print()
print(f'Model E Test Results:')
print(f'  Macro F1    : {test_macro_e:.4f}')
print(f'  Minority F1 : {test_minor_e:.4f}')
print(f'  Majority F1 : {test_major_e:.4f}')
print(f'  AUC-ROC     : {test_auc_e:.4f}')
print(f'  Accuracy    : {test_accuracy_e:.4f}')


## 9. Three-Way Comparison: Model D vs TF-IDF vs Model E

This ablation answers: **does SBERT specifically add value, or does any text
representation help?**

We compare three approaches:
1. **Model D** — T4 structured features only (no text)
2. **TF-IDF fusion** — T4 + TF-IDF text features (bag-of-words baseline)
3. **Model E** — T4 + SBERT PCA50 (semantic embeddings)

If Model E outperforms TF-IDF, it confirms that semantic understanding
(not just word frequency) is what drives the improvement.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# ── Model D baseline (structured only — no text) ──────────────────────────────
print('Evaluating Model D baseline (structured only)...')
clf_d = build_catboost()
clf_d.fit(X_t4_train, y_train)
y_pred_d = clf_d.predict(X_t4_test)
y_prob_d = clf_d.predict_proba(X_t4_test)[:, 1]

test_macro_d = f1_score(y_test, y_pred_d, average='macro')
test_minor_d = f1_score(y_test, y_pred_d, pos_label=0)
test_auc_d      = roc_auc_score(y_test, y_prob_d)
test_accuracy_d = accuracy_score(y_test, y_pred_d)
print(f'  Model D: Macro F1={test_macro_d:.4f}  Minority F1={test_minor_d:.4f}  AUC={test_auc_d:.4f}  Accuracy={test_accuracy_d:.4f}')

# ── TF-IDF fusion ─────────────────────────────────────────────────────────────
# TF-IDF: counts word frequencies, no semantic understanding
# We reduce to 50 dims with TruncatedSVD (LSA) to match SBERT PCA50 dimensionality
print('Building TF-IDF + LSA text features...')

desc_train = descriptions[train_idx]
desc_test  = descriptions[test_idx]

# Replace empty descriptions with empty string for TF-IDF
desc_train_clean = [d if d else '' for d in desc_train]
desc_test_clean  = [d if d else '' for d in desc_test]

tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
tfidf_train = tfidf.fit_transform(desc_train_clean)  # fit on train only
tfidf_test  = tfidf.transform(desc_test_clean)

# Reduce TF-IDF to 50 dims with LSA (Latent Semantic Analysis)
lsa = TruncatedSVD(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
tfidf_train_50 = lsa.fit_transform(tfidf_train)  # fit on train only
tfidf_test_50  = lsa.transform(tfidf_test)

# Concatenate structured + TF-IDF
X_tfidf_train = np.hstack([X_t4_train, tfidf_train_50])
X_tfidf_test  = np.hstack([X_t4_test,  tfidf_test_50])

print('Training CatBoost on T4 + TF-IDF...')
clf_tfidf = build_catboost()
clf_tfidf.fit(X_tfidf_train, y_train)
y_pred_tfidf = clf_tfidf.predict(X_tfidf_test)
y_prob_tfidf = clf_tfidf.predict_proba(X_tfidf_test)[:, 1]

test_macro_tfidf = f1_score(y_test, y_pred_tfidf, average='macro')
test_minor_tfidf = f1_score(y_test, y_pred_tfidf, pos_label=0)
test_auc_tfidf      = roc_auc_score(y_test, y_prob_tfidf)
test_accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)
print(f'  TF-IDF: Macro F1={test_macro_tfidf:.4f}  Minority F1={test_minor_tfidf:.4f}  AUC={test_auc_tfidf:.4f}  Accuracy={test_accuracy_tfidf:.4f}')

# ── Three-way comparison table ─────────────────────────────────────────────────
print()
print('=== THREE-WAY COMPARISON ===')
print(f'  {"Approach":<30} {"Macro F1":>10} {"Minority F1":>12} {"AUC-ROC":>10} {"Accuracy":>10}')
print('  ' + '-' * 78)
for label, macro, minor, auc, acc in [
    ('Model D (structured only)',    test_macro_d,     test_minor_d,     test_auc_d,     test_accuracy_d),
    ('TF-IDF + LSA (50 dims)',       test_macro_tfidf, test_minor_tfidf, test_auc_tfidf, test_accuracy_tfidf),
    ('Model E (SBERT PCA50)',        test_macro_e,     test_minor_e,     test_auc_e,     test_accuracy_e),
]:
    print(f'  {label:<30} {macro:>10.4f} {minor:>12.4f} {auc:>10.4f} {acc:>10.4f}')

print()
sbert_vs_d     = test_macro_e - test_macro_d
sbert_vs_tfidf = test_macro_e - test_macro_tfidf
print(f'SBERT gain over Model D    : Macro F1 {sbert_vs_d:+.4f}')
print(f'SBERT gain over TF-IDF     : Macro F1 {sbert_vs_tfidf:+.4f}')

# ── Visualise comparison ───────────────────────────────────────────────────────
labels    = ['Model D\n(no text)', 'TF-IDF\n+LSA', 'Model E\n(SBERT PCA50)']
macros    = [test_macro_d,     test_macro_tfidf,    test_macro_e]
minors    = [test_minor_d,     test_minor_tfidf,    test_minor_e]
aucs_cmp  = [test_auc_d,       test_auc_tfidf,      test_auc_e]
accs_cmp  = [test_accuracy_d,  test_accuracy_tfidf, test_accuracy_e]
colors    = ['#94a3b8', '#f59e0b', '#3b82f6']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Three-Way Comparison: Text Encoding Strategies (Test Set)', fontsize=13)

axes[0].bar(labels, macros, color=colors, edgecolor='white')
axes[0].set_title('Macro F1 (PRIMARY)')
axes[0].set_ylim(0.55, 0.85)
for i, v in enumerate(macros):
    axes[0].text(i, v + 0.003, f'{v:.4f}', ha='center', fontsize=9)

axes[1].bar(labels, minors, color=colors, edgecolor='white')
axes[1].set_title('Minority F1 (Not Well Received)')
axes[1].set_ylim(0.4, 0.75)
for i, v in enumerate(minors):
    axes[1].text(i, v + 0.003, f'{v:.4f}', ha='center', fontsize=9)

axes[2].bar(labels, aucs_cmp, color=colors, edgecolor='white')
axes[2].set_title('AUC-ROC')
axes[2].set_ylim(0.65, 0.85)
for i, v in enumerate(aucs_cmp):
    axes[2].text(i, v + 0.003, f'{v:.4f}', ha='center', fontsize=9)

axes[3].bar(labels, accs_cmp, color=colors, edgecolor='white')
axes[3].set_title('Accuracy')
axes[3].set_ylim(0.55, 0.85)
for i, v in enumerate(accs_cmp):
    axes[3].text(i, v + 0.003, f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/05_three_way_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/05_three_way_comparison.png')


## 10. Ablation — Description Coverage Impact

37 games (0.2%) have no description. These receive a zero vector.
This section verifies that the zero-vector approach does not harm Model E's
performance for games that DO have descriptions.

We also confirm the routing rule: games with no description → Model D fallback.


In [ ]:
# ── Check: are the 37 missing-description games in train or test? ─────────────
empty_mask_train = (descriptions[train_idx] == '')
empty_mask_test  = (descriptions[test_idx]  == '')

print(f'Missing descriptions in train: {empty_mask_train.sum()}')
print(f'Missing descriptions in test : {empty_mask_test.sum()}')

# ── Evaluate Model E on test games WITH descriptions only ─────────────────────
has_desc_test = ~empty_mask_test
y_test_with   = y_test[has_desc_test]
X_t5_test_with = X_t5_test[has_desc_test]

y_pred_e_with   = clf_e_final.predict(X_t5_test_with)
test_macro_with = f1_score(y_test_with, y_pred_e_with, average='macro')
test_minor_with = f1_score(y_test_with, y_pred_e_with, pos_label=0)

print()
print(f'Model E performance on games WITH descriptions ({has_desc_test.sum():,} games):')
print(f'  Macro F1    : {test_macro_with:.4f}')
print(f'  Minority F1 : {test_minor_with:.4f}')
print()
print(f'Model E performance on ALL test games ({len(y_test):,} games, incl. zero-vector):')
print(f'  Macro F1    : {test_macro_e:.4f}')
print(f'  Minority F1 : {test_minor_e:.4f}')
print()
print('Router rule confirmed:')
print(f'  Games with description    ({has_desc_test.sum():,}): route to Model E')
print(f'  Games without description ({empty_mask_test.sum():,}): route to Model D')

# ── Model D performance on the 37 no-description test games (if any) ─────────
if empty_mask_test.sum() > 0:
    y_test_empty   = y_test[empty_mask_test]
    X_t4_test_empty = X_t4_test[empty_mask_test]
    y_pred_d_empty  = clf_d.predict(X_t4_test_empty)
    if len(np.unique(y_test_empty)) > 1:
        macro_empty = f1_score(y_test_empty, y_pred_d_empty, average='macro')
        print(f'  Model D on no-description games: Macro F1={macro_empty:.4f}')
    else:
        print(f'  No-description test games all same class — F1 not meaningful')


## 11. Save All Outputs

Save the four output files to `/content/` then copy them to Google Drive.
After the notebook completes, download them to your local machine.

**Local placement:**
- `sbert_embeddings_raw.npy` → `data/processed/`
- `sbert_embeddings_pca{N}.npy` → `data/processed/`  *(N = PCA_COMPONENTS selected in 6a)*
- `pca_transformer.pkl` → `models/`
- `model_e.pkl` → `models/`


In [ ]:
import os, shutil

# ── Save Model E ───────────────────────────────────────────────────────────────
joblib.dump(clf_e_final, '/content/model_e.pkl')
print('Saved: /content/model_e.pkl')

# ── Save ablation results ──────────────────────────────────────────────────────
import pandas as pd

# PCA component search results
pca_search_rows = []
for n in sorted(performance_results.keys()):
    pca_search_rows.append({
        'pca_components': n,
        'variance_retained': round(variance_results[n], 2),
        'cv_macro_f1_3fold': round(performance_results[n], 4),
        'selected': n == PCA_COMPONENTS
    })
pd.DataFrame(pca_search_rows).to_csv('/content/pca_search_results.csv', index=False)
print('Saved: /content/pca_search_results.csv')

# Three-way text ablation
ablation_rows = [
    {'approach': 'Model D (structured only)', 'text_features': 'none',
     'pca_components': 0,
     'test_macro_f1': round(test_macro_d, 4),
     'test_minority_f1': round(test_minor_d, 4),
     'test_auc': round(test_auc_d, 4),
     'test_accuracy': round(test_accuracy_d, 4)},
    {'approach': 'TF-IDF + LSA (50 dims)', 'text_features': 'TF-IDF',
     'pca_components': 50,
     'test_macro_f1': round(test_macro_tfidf, 4),
     'test_minority_f1': round(test_minor_tfidf, 4),
     'test_auc': round(test_auc_tfidf, 4),
     'test_accuracy': round(test_accuracy_tfidf, 4)},
    {'approach': f'Model E (SBERT PCA{PCA_COMPONENTS})', 'text_features': 'SBERT',
     'pca_components': PCA_COMPONENTS,
     'test_macro_f1': round(test_macro_e, 4),
     'test_minority_f1': round(test_minor_e, 4),
     'test_auc': round(test_auc_e, 4),
     'test_accuracy': round(test_accuracy_e, 4)},
]
pd.DataFrame(ablation_rows).to_csv('/content/sbert_ablation.csv', index=False)
print('Saved: /content/sbert_ablation.csv')

# ── Copy everything to Google Drive ───────────────────────────────────────────
DRIVE_OUTPUT = '/content/drive/MyDrive/fyp_pprs/outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Dynamic filenames based on selected PCA_COMPONENTS
files_to_copy = [
    '/content/sbert_embeddings_raw.npy',
    f'/content/{PCA_NPY_FILENAME}',           # e.g. sbert_embeddings_pca100.npy
    '/content/pca_transformer.pkl',
    '/content/model_e.pkl',
    '/content/sbert_ablation.csv',
    '/content/pca_search_results.csv',        # new — PCA search results
    '/content/05_pca_variance.png',
    '/content/05_three_way_comparison.png',
]

print()
print('Copying files to Google Drive...')
for src in files_to_copy:
    if os.path.exists(src):
        dst = os.path.join(DRIVE_OUTPUT, os.path.basename(src))
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  Copied: {os.path.basename(src)}  ({size_mb:.1f} MB)')
    else:
        print(f'  MISSING: {src}')

print()
print(f'All files saved to: {DRIVE_OUTPUT}')
print(f'Embedding file saved as: {PCA_NPY_FILENAME}')
print(f'Update config.py: SBERT_PCA_NPY = DATA_PROCESSED / "{PCA_NPY_FILENAME}"')


## 12. Summary & Download Instructions

In [ ]:
print('=' * 65)
print('NOTEBOOK 05 — MODEL E SBERT FUSION SUMMARY')
print('=' * 65)

print(f'\nSBERT ENCODING')
print(f'  Model              : {SBERT_MODEL}')
print(f'  Input dim          : 384')
print(f'  PCA selection      : empirical (best CV macro F1 across candidates)')
print(f'  PCA_COMPONENTS     : {PCA_COMPONENTS}  (variance={variance_retained:.1f}%)')
print(f'  Encoding device    : {DEVICE.upper()}')

print(f'\nPCA COMPONENT SEARCH RESULTS:')
print(f'  {"n":>6}  {"Variance":>10}  {"CV Macro F1":>12}')
print('  ' + '-' * 34)
for n in sorted(performance_results.keys()):
    marker = ' ← selected' if n == PCA_COMPONENTS else ''
    print(f'  {n:>6}  {variance_results[n]:>9.1f}%  {performance_results[n]:>12.4f}{marker}')

print(f'\nT5 FEATURE MATRIX')
print(f'  Structured (T4)    : {X_t4_train.shape[1]} features')
print(f'  SBERT PCA dims     : {PCA_COMPONENTS} features')
print(f'  Total T5           : {X_t5_train.shape[1]} features')

print(f'\nMODEL E RESULTS (5-fold CV + test set)')
print(f'  CV  Macro F1       : {scores_e["test_f1_macro"].mean():.4f} ± {scores_e["test_f1_macro"].std():.4f}')
print(f'  CV  Minority F1    : {scores_e["test_f1_minority"].mean():.4f}')
print(f'  CV  Accuracy       : {scores_e["test_accuracy"].mean():.4f} ± {scores_e["test_accuracy"].std():.4f}')
print(f'  Test Macro F1      : {test_macro_e:.4f}')
print(f'  Test Minority F1   : {test_minor_e:.4f}')
print(f'  Test Accuracy      : {test_accuracy_e:.4f}')
print(f'  Test AUC-ROC       : {test_auc_e:.4f}')

print(f'\nTHREE-WAY COMPARISON (test set):')
print(f'  {"Approach":<30} {"Macro F1":>10} {"Minority F1":>12} {"AUC":>8} {"Accuracy":>10}')
print('  ' + '-' * 76)
print(f'  {"Model D (structured only)":<30} {test_macro_d:>10.4f} {test_minor_d:>12.4f} {test_auc_d:>8.4f} {test_accuracy_d:>10.4f}')
print(f'  {"TF-IDF + LSA (50 dims)":<30} {test_macro_tfidf:>10.4f} {test_minor_tfidf:>12.4f} {test_auc_tfidf:>8.4f} {test_accuracy_tfidf:>10.4f}')
print(f'  {f"Model E (SBERT PCA{PCA_COMPONENTS})":<30} {test_macro_e:>10.4f} {test_minor_e:>12.4f} {test_auc_e:>8.4f} {test_accuracy_e:>10.4f}')
print(f'\n  SBERT gain over Model D    : Macro F1 {test_macro_e - test_macro_d:+.4f}')
print(f'  SBERT gain over TF-IDF     : Macro F1 {test_macro_e - test_macro_tfidf:+.4f}')

print(f'\nDESCRIPTION COVERAGE')
print(f'  With description   : {(~empty_mask_test).sum() + (~empty_mask_train).sum():,} ({coverage:.1f}%)')
print(f'  Without (→ Model D): {empty_mask_test.sum() + empty_mask_train.sum():,}')

print(f'\n{"="*65}')
print('DOWNLOAD FROM GOOGLE DRIVE')
print(f'{"="*65}')
print(f'Drive path: /content/drive/MyDrive/fyp_pprs/outputs')
print()
print(f'  sbert_embeddings_raw.npy    → data/processed/')
print(f'  {PCA_NPY_FILENAME:<35} → data/processed/')
print(f'  pca_transformer.pkl         → models/')
print(f'  model_e.pkl                 → models/')
print(f'  sbert_ablation.csv          → outputs/results/')
print(f'  pca_search_results.csv      → outputs/results/')
print(f'  05_pca_variance.png         → outputs/figures/')
print(f'  05_three_way_comparison.png → outputs/figures/')
print()
print(f'UPDATE config.py:')
print(f'  SBERT_PCA_NPY = DATA_PROCESSED / "{PCA_NPY_FILENAME}"')
print()
print('NEXT STEP: Notebook 06 — SHAP + LIME Explainability (run locally)')
print('  All 5 models must be in models/ before running notebook 06')
print('=' * 65)
